# 623 SPP v24: natural-cardinality LSTM sweep

This notebook reuses the v23 input archive unchanged. It first compares global and event-routed chronological recurrent cores at h32 using GUARD natural action-list NLL only. It then trains h8/16/32/64/128 with the selected core. Each final model predicts a natural categorical count and exactly that many rank-conditioned TRAIN-observed `(delta, fill)` actions. Teacher actions are labels/comparators only; there is no STOP padding, class weighting, prior correction, action feedback, request budget, or page rule.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import drive, files, userdata
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), f'Select an A100 runtime; observed {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets and allow notebook access'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/train_and_offline_infer.py'
CONTRACT_SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/model_contract.py'
VALIDATOR=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/validate_collected_inputs.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--describe-model-points'],text=True))
assert MODEL_CONTRACT==json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
RUN_ID=MODEL_CONTRACT['run_id']; TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; TRAINING=MODEL_CONTRACT['training_config']
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/cache_prefetch_623_spp')/RUN_ID; DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
print('Select',f'{RUN_ID}.colab_input.tar.gz')
uploaded=files.upload(); archives=[pathlib.Path('/content',name) for name in uploaded if name.endswith('.colab_input.tar.gz')]
assert len(archives)==1, 'Upload exactly one .colab_input.tar.gz'
INPUT_DIR=pathlib.Path(f'/content/{RUN_ID}_colab_input')
if INPUT_DIR.exists(): shutil.rmtree(INPUT_DIR)
sys.path.insert(0,f'{REPO}/formal_NN_training/common')
import split_colab_archive as transfer
transfer.safe_extract_tar_gz(archives[0],INPUT_DIR)
transfer.validate_sha256sums(INPUT_DIR)
SOURCE=INPUT_DIR/'spp_source_contract.json'; assert SOURCE.is_file()
ROLES=('train','guard','eval')
INPUTS={role:{'stream':INPUT_DIR/f'{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':INPUT_DIR/f'{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
    for path in items.values(): assert path.is_file(),path
VALIDATED=pathlib.Path('/content/validated_collection_manifest.json')
subprocess.run([sys.executable,VALIDATOR,'--input-dir',str(INPUT_DIR),'--manifest-out',str(VALIDATED),'--source-contract',str(SOURCE)],check=True)
manifest=json.loads(VALIDATED.read_text())
assert manifest['status']=='PASS' and manifest['source_decision_effective_external_input']==MODEL_CONTRACT['external_input_fields']
assert manifest['stop_padding_used'] is False and manifest['loss_class_reweighting_used'] is False and manifest['decode_prior_correction_used'] is False
print('fresh matched-input validation PASS')

In [ ]:
LOCAL_OUTPUT=DRIVE_ROOT/'colab_output'; LOCAL_OUTPUT.mkdir(exist_ok=True)
shutil.copy2(VALIDATED,LOCAL_OUTPUT/'validated_collection_manifest.json')
LOG_ROOT=DRIVE_ROOT/'trainer_logs'; LOG_ROOT.mkdir(exist_ok=True)
def run_logged(cmd,log_path):
    print('running',cmd[-5:] if len(cmd)>5 else cmd,flush=True)
    with pathlib.Path(log_path).open('w',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,cmd)
def common_cmd(size,pair_id,core,mode,out):
    cmd=[sys.executable,SCRIPT,'--policy',POLICY,'--train-stream',str(INPUTS['train']['stream']),'--train-teacher-actions',str(INPUTS['train']['teacher']),'--guard-stream',str(INPUTS['guard']['stream']),'--guard-teacher-actions',str(INPUTS['guard']['teacher']),'--source-contract',str(SOURCE),'--out-dir',str(out),'--model-family','lstm','--model-size',str(size),'--pair-id',pair_id,'--run-mode',mode,'--core-type',core,'--device','cuda','--seed',str(TRAINING['seed']),'--epochs',str(TRAINING['epochs']),'--chunk-len',str(TRAINING['chunk_len']),'--accumulate-chunks',str(TRAINING['accumulate_chunks']),'--learning-rate',str(TRAINING['learning_rate'])]
    return cmd
assert MODEL_CONTRACT['core_selection_uses_evaluation'] is False
h32=next(point for point in MODEL_CONTRACT['points'] if point['size']==MODEL_CONTRACT['core_selection_hidden_size'])
ABLATION_ROOT=DRIVE_ROOT/'core_ablation'; ABLATION_ROOT.mkdir(exist_ok=True)
candidates={}
for core in MODEL_CONTRACT['core_types']:
    out=ABLATION_ROOT/core
    cmd=common_cmd(h32['size'],h32['pair_id'],core,'core-ablation',out)
    run_logged(cmd,LOG_ROOT/f'core_ablation_{core}.log')
    meta=json.loads((out/'run_metadata.json').read_text())
    assert meta['run_mode']=='core-ablation' and meta['evaluation_files_loaded'] is False and meta['core_type']==core
    candidates[core]=meta['selected_guard_natural_action_list_nll']['natural_action_list_nll_per_callback']
selected=min(MODEL_CONTRACT['core_types'],key=lambda core:(candidates[core],core!=MODEL_CONTRACT['core_selection_tie_break']))
CORE_SELECTION={'status':'PASS','selection_hidden_size':MODEL_CONTRACT['core_selection_hidden_size'],'selection_metric':MODEL_CONTRACT['core_selection_metric'],'tie_break':MODEL_CONTRACT['core_selection_tie_break'],'selected_core':selected,'candidates':candidates,'evaluation_used':False}
CORE_SELECTION_PATH=DRIVE_ROOT/'spp_v24_core_selection.json'; CORE_SELECTION_PATH.write_text(json.dumps(CORE_SELECTION,indent=2,sort_keys=True)+'\n')
print('selected core',selected,candidates)

In [ ]:
POINTS=sorted(MODEL_CONTRACT['points'],key=lambda point:point['size'])
assert [point['size'] for point in POINTS]==[8,16,32,64,128]
SWEEP=[]
for point in POINTS:
    out=LOCAL_OUTPUT/point['tag']
    cmd=common_cmd(point['size'],point['pair_id'],selected,'final',out)
    cmd += ['--eval-stream',str(INPUTS['eval']['stream']),'--eval-teacher-actions',str(INPUTS['eval']['teacher']),'--core-selection-file',str(CORE_SELECTION_PATH)]
    log_path=LOG_ROOT/f"{point['tag']}.log"
    run_logged(cmd,log_path); shutil.copy2(log_path,out/'trainer.stdout_stderr.log')
    meta=json.loads((out/'run_metadata.json').read_text())
    required={'run_id':RUN_ID,'run_mode':'final','model_tag':point['tag'],'model_size':point['size'],'architecture_pair_id':point['pair_id'],'core_type':selected,'selected_core_type':selected,'teacher_actions_are_model_inputs':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_request_rate_used_as_budget':False,'probability_threshold_used':False,'inference_policy_hardcodes_used':False,'stop_padding_used':False,'loss_class_reweighting_used':False,'decode_prior_correction_used':False,'evaluation_used_for_selection':False,'evaluation_policy_decode_count':1,'oracle_diagnostics_replayed':False,'non_neural_control_excluded_from_neural_claims':True}
    bad={key:(meta.get(key),value) for key,value in required.items() if meta.get(key)!=value}; assert not bad,bad
    for name in ('run_metadata.json','training_history.csv','model.pt','offline_spp.replay.csv','offline_nn.replay.csv','offline_modal_llc_control.replay.csv','trainer.stdout_stderr.log'): assert (out/name).is_file(),(out/name)
    SWEEP.append({'model_tag':point['tag'],'model_size':point['size'],'pair_id':point['pair_id'],'core_type':selected,'parameter_count':meta['parameter_count'],'selected_epoch':meta['selected_epoch'],'guard_natural_action_list_nll':meta['selected_guard_natural_action_list_nll']['natural_action_list_nll_per_callback'],'offline_normal_entries':meta['offline_normal_entries'],'offline_nn_entries':meta['offline_nn_entries'],'nn_list_sha256':meta['nn_list_sha256']})
SWEEP_MANIFEST={'status':'PASS','run_id':RUN_ID,'trace':TRACE,'policy':POLICY,'fresh_input_validation_manifest':'validated_collection_manifest.json','core_selection':CORE_SELECTION,'points':SWEEP}
(LOCAL_OUTPUT/'sweep_manifest.json').write_text(json.dumps(SWEEP_MANIFEST,indent=2,sort_keys=True)+'\n')
assert len(SWEEP)==5
print(json.dumps(SWEEP_MANIFEST,indent=2))

In [ ]:
OUTPUT_ARCHIVE=pathlib.Path(f'/content/{RUN_ID}.colab_output.tar.gz')
if OUTPUT_ARCHIVE.exists(): OUTPUT_ARCHIVE.unlink()
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
    for item in sorted(LOCAL_OUTPUT.iterdir(),key=lambda path:path.name): archive.add(item,arcname=item.name)
with tarfile.open(OUTPUT_ARCHIVE,'r:gz') as archive: assert archive.getmembers()
print('downloading',OUTPUT_ARCHIVE.name,OUTPUT_ARCHIVE.stat().st_size,'bytes')
files.download(str(OUTPUT_ARCHIVE))

The archive contains only the five final neural points, the fresh input-validation manifest, and the sweep manifest containing the GUARD-only core-selection evidence. The h32 architecture candidates are diagnosis-only and are not replayed. The modal-LLC policy remains a separate non-neural control and cannot support a neural win claim.